LOGGING E MONITORING MODELLI IN PRODUZIONE

Come tener d'occhio un modello AI quanto è in produzione.
Il modello va sempre monitorato
- System Metrics: monitori l'hardware (CPU, RAM, GPU) con Prometheus, ogni 15 secondi fa un check tipo "tutto bene?" e registra tutto in un database temporale.
- Model Metrics: con Grafana visualizzi cosa sta dicendo il modello: confidence delle predizioni, distribuzione delle classi, entropia (quanto è confuso l'AI)
- Data Drift: il nemico subdolo: il mondo cambia, i data si spostano, e il modello invecchia. Lo misuri con la divergenza di Kullback-Leibler e lo combatti con retraining continuo e human-in-the-loop

SYSTEM METRICS
Metafora: puoi avere il miglior pilota AI del mondo, ma se il motore (la GPU) si surriscalda o finisce la memoria, la gara è finita
Dobbiamo monitorare l'ambiente di esecuzione.
Prometheus è come un supervisore che ogni 15 secondi chiede 'come va?' Va a leggere i dati da un endpoint/metrics che il codice Python espone (es. FastAPI)
Raccoglie dati direttamente dagli esportatori che interrogano direttamente l'hardware
Le 3 cose da tenere d'occhio:
- GPU Utilization: il modello sta lavorando o se la dorme? Percentuale di tempo in cui i tensori sono attivi. Un valore costantemente al 100% indica la necessità di scalare orizzontalmente i worker
- VRAM Occupancy: se la memoria sale e non scende mai, hai un memory leak in arrivo
- P99 Latency: tempo di attesa dell'utente più sfortunato (99-esimo percentile). Se è alto, il sistema sta soffrendo
Se mandi dati uno alla volta, la latenza è bassa ma la GPU è sprecata. Se fai batch, la GPU è felice ma il primo a ricevere risposta aspetta di più. E' necessario trovare il punto di equilibrio, throughput optimization

Come fa Prometheus ad ottenere questi dati? 
Interroga il server AI ad intervalli regolari (es. ogni 15 secondi) leggendo le metriche Per far questo interroga un endpoint speciale, solitamente  /metrics dove FastAPI espone i dati grezzi.
I dati sono salvati come sequenze di numeri nel tempo. Questo permette di creare grafici storici per identificare se la latenza aumenta durante le ore di punta.
E' necessario configurare delle soglie critiche (alert manager). Esempio inviare un messaggio Slack, o una mail, se la temperatura della GPU supera gli 80 gradi o se la memoria RAM è satura al 90%

MODEL METRICS
Se  l'hardware sta benissimo (CPU al 30%, GPU felice), il modello potrebbe comunque dire stupidate. E l'utente accorgersene prima del tuo cluster.
Oltre all'hardware dobbiamo monitorare 'cosa' il modello sta dicendo. Se un classificatore di sentiment inizia a predire solo 'negativo' c'è un problema statistico che le metriche di sistema non rilevano mai.
Grafana si collega a Prometheus per trasformare serie temporali astratte in dashboard intuitive, permettendo di correlare le predizioni dell'AI con i risultati economici reali.
Le metriche chiave da monitorare con Grafana:
- Prediction Confidence: se la softmax media del modello crolla, vuol dire che l'AI non è più sicura di ciò che vede. Se la confidenza media cala drasticamente, il modello potrebbe trovarsi davanti ai dati nuovi
- Class distribution: se in un'ora il classificatore di mail rileva più frodi di quante ne abbia rilevate in un anno, qualcosa è rotto. Conteggio delle occorrenze per ogni classe predetta.
- Entropia: il termometro dell'esitazione. Più è alta, più il modello sta tirando a indovinare. Misura la confusione totale del modello. Un AI confusa ha bisogno di manutenzione.
L'entropia la puoi calcolare senza sapere la risposta giusta, è un segnale premonitore che arriva giorni prima che l'accuratezza crolli.
Tecnicamente guardi la distribuzione di probabilità in uscita dalla softmax:
- Molto sicuro: entropia bassa o 0
- Confuso: entropia alta
- Completamente perso: entropia massima
Su Grafana puoi tracciare l'entropia media nel tempo. Se vedi un'impennata, sai che c'è data drift in arrivo prima ancora di sapere cosa è cambiato.

Visualizzaizone con Grafana
- Dashboard Interattive: creazione di pannelli che mostrano la 'headmap' della latenza e grafici a torta sulla distribuzione delle label predette null'ultima ora.
- Corrrelazione delle sorgenti: possibilità di sovrapporre il grafico del carico CPU con quello della latenza per confermare che i rallentamenti siano dovuti a limiti hardware
- Accessi Multi-utente: configurazione di permessi diversi per ingegneri (che guardano la CPU) e production manager (che guardano la qualità delle risposte dell'LLM)
Dashboard diverse per persone diverse.

DATA DRIFT
Data drift è il motivo per cui l'entropia sale. Il mondo cambia ed il modello invecchia. Un modello è una fotografia perfetta per il momento in cui è stata scattata. Ma le persone invecchiano, cambiano vestiti, cambiano gusti.
Se addestri un modello per riconoscere spam oggi, tra due anni gli spanner avranno inventato nuovi trucchi che il tuo modello non ha mai visto. E' il divorzio statistico tra la realtà e il tuo training set.
Due tipi:
- Feature drift:  cambiano gli input. Esempio gli utenti iniziano a fotografare con gli smartphone nuovi che hanno colori diversi. I pixel in input sono spostati.
- Concept drift: cambia il significato. una parole che ieri era un complimento, oggi potrebbe essere un insulto. Il modello non lo sa.
Come lo misuri? con la divergenza di Kullback-Leibler, un termometro statistico che dice quanto la distribuzione attuale Q è diversa da quella originale P, su cui il modello è stato addestrato.
Tracciando queste metriche, possiamo attivare allert prima che l'accuratezza reali cali visibilmente.
Se il valore supera un soglia, il modello è ufficialmente scaduto.
E la cura? continuous retraining, quando Prometheus rileva drift oltre la soglia, innesca automaticamente una pipeline che raccoglie dati freschi (esempio raccoglie i dati dell'ultima settimana=) e riaddestra il modello. Con human-in-the-loop sui casi più incerti, affinchè ci dia la verità corretta.
Nei modelli moderni possiamo monitorare il drift anche quando non vediamo i dati originali.
Invece di guardare la parole, guardiamo gli embedding, Se gli embedding iniziano a finire in una zona vuota, il modello sta incontrando dati isolati e non addestrati ed il modello tira ad indovinare.


